# SkipperNDT — Tâche 2 : Régression map_width (GPU Colab)

**Objectif** : MAE < 1 mètre sur la prédiction de la largeur de carte magnétique.

**Stratégie** :
- Entraînement en log-space (log1p/expm1) pour la distribution asymétrique (2m→155m)
- Huber loss robuste aux outliers
- MagCNN puis MagDenseNet si nécessaire
- Test final sur les 51 données réelles avec width_m connu

**Setup** : Runtime → Modifier le type d'exécution → GPU (T4)

In [ ]:
# ── 0. Vérification GPU ───────────────────────────────────────
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── 1. Montage Google Drive ───────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
# Adapter ce chemin selon où tu as mis les données sur ton Drive
DRIVE_DATA = '/content/drive/MyDrive/skipper_ndt/Training_database_float16'
CSV_PATH   = '/content/drive/MyDrive/skipper_ndt/pipe_presence_width_detection_label.csv'

print(f'Données : {os.path.exists(DRIVE_DATA)}')
print(f'CSV     : {os.path.exists(CSV_PATH)}')
if os.path.exists(DRIVE_DATA):
    n = len([f for f in os.listdir(DRIVE_DATA) if f.endswith('.npz')])
    print(f'Fichiers .npz : {n}')

In [ ]:
# ── 2. Imports ────────────────────────────────────────────────
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from scipy.ndimage import zoom
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')

In [ ]:
# ── 3. Preprocessing ──────────────────────────────────────────

def load_npz(path):
    """Charge un fichier .npz et retourne l'array (H, W, 4) float32."""
    d = np.load(path)['data'].astype('float32')
    return d

def resize_array(arr, size=128):
    """Resize vers (size, size, 4) avec scipy zoom, préserve les NaN."""
    H, W, C = arr.shape
    zh, zw = size / H, size / W
    out = np.zeros((size, size, C), dtype='float32')
    for c in range(C):
        ch = arr[:, :, c]
        mask = np.isnan(ch)
        ch_filled = np.where(mask, 0.0, ch)
        out[:, :, c] = zoom(ch_filled, (zh, zw), order=1)
    return out

def normalize_channels(arr):
    """Zscore individuel par canal, NaN → 0."""
    out = arr.copy()
    for c in range(arr.shape[2]):
        ch = out[:, :, c]
        valid = ch[ch != 0.0]
        if len(valid) > 0:
            mu, sigma = valid.mean(), valid.std()
            if sigma > 1e-8:
                out[:, :, c] = np.where(ch != 0.0, (ch - mu) / sigma, 0.0)
    return out

print('✓ Preprocessing défini')

In [ ]:
# ── 4. Dataset ────────────────────────────────────────────────

class RegressionDataset(Dataset):
    def __init__(self, paths, widths, augment=False, size=128):
        self.paths   = paths
        self.targets = [float(np.log1p(w)) for w in widths]  # log-space
        self.augment = augment
        self.size    = size

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        arr = load_npz(self.paths[idx])
        arr = resize_array(arr, self.size)
        arr = normalize_channels(arr)

        if self.augment:
            if np.random.rand() > 0.5:
                arr = arr[:, ::-1, :].copy()
            if np.random.rand() > 0.5:
                arr = arr[::-1, :, :].copy()
            k = np.random.randint(0, 4)
            if k > 0:
                arr = np.rot90(arr, k=k, axes=(0, 1)).copy()

        tensor = torch.from_numpy(arr.transpose(2, 0, 1))
        target = torch.tensor(self.targets[idx], dtype=torch.float32)
        return tensor, target

print('✓ Dataset défini')

In [ ]:
# ── 5. Architectures ──────────────────────────────────────────

class MagCNNRegressor(nn.Module):
    """CNN léger de régression ~430K params."""
    def __init__(self, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(4,  32, 3, padding=1, bias=False), nn.BatchNorm2d(32),  nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64),  nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(64,128, 3, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(128,256,3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(True),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128), nn.ReLU(True), nn.Dropout(dropout),
            nn.Linear(128,  64), nn.ReLU(True), nn.Dropout(dropout/2),
            nn.Linear(64,    1),
        )
    def forward(self, x):
        return self.gap(self.features(x)).flatten(1).squeeze(1) if False else self.regressor(self.gap(self.features(x)))

# Version corrigée
class MagCNNRegressor(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(4,  32, 3, padding=1, bias=False), nn.BatchNorm2d(32),  nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64),  nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(64,128, 3, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(128,256,3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(256,128), nn.ReLU(True), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(True), nn.Dropout(dropout/2),
            nn.Linear(64,   1),
        )
    def forward(self, x):
        return self.net(x).squeeze(1)


class MagDenseNetRegressor(nn.Module):
    """DenseNet de régression ~1.8M params."""
    def __init__(self, growth_rate=16, dropout=0.3):
        super().__init__()
        k = growth_rate
        self.stem   = nn.Sequential(
            nn.Conv2d(4, 2*k, 3, padding=1, bias=False),
            nn.BatchNorm2d(2*k), nn.ReLU(True), nn.MaxPool2d(2)
        )
        self.dense1 = self._make_dense(2*k,       6, k)
        n1 = 2*k + 6*k
        self.trans1 = self._make_trans(n1, n1//2)
        self.dense2 = self._make_dense(n1//2,     6, k)
        n2 = n1//2 + 6*k
        self.trans2 = self._make_trans(n2, n2//2)
        self.dense3 = self._make_dense(n2//2,     6, k)
        n3 = n2//2 + 6*k
        self.bn     = nn.BatchNorm2d(n3)
        self.gap    = nn.AdaptiveAvgPool2d(1)
        self.head   = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n3,128), nn.ReLU(True), nn.Dropout(dropout),
            nn.Linear(128,  1)
        )
        self.n_out = n3

    @staticmethod
    def _make_dense(in_ch, n, k):
        layers = nn.ModuleList()
        ch = in_ch
        for _ in range(n):
            layers.append(nn.Sequential(
                nn.BatchNorm2d(ch), nn.ReLU(True),
                nn.Conv2d(ch, k, 3, padding=1, bias=False)
            ))
            ch += k
        return layers

    @staticmethod
    def _make_trans(in_ch, out_ch):
        return nn.Sequential(
            nn.BatchNorm2d(in_ch), nn.ReLU(True),
            nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.AvgPool2d(2)
        )

    def _fwd_dense(self, x, block):
        feats = [x]
        for layer in block:
            feats.append(layer(torch.cat(feats, 1)))
        return torch.cat(feats, 1)

    def forward(self, x):
        x = self.stem(x)
        x = self._fwd_dense(x, self.dense1)
        x = self.trans1(x)
        x = self._fwd_dense(x, self.dense2)
        x = self.trans2(x)
        x = self._fwd_dense(x, self.dense3)
        x = F.relu(self.bn(x))
        return self.head(self.gap(x)).squeeze(1)


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

cnn  = MagCNNRegressor()
dn   = MagDenseNetRegressor()
print(f'MagCNNRegressor     : {count_params(cnn):,} params')
print(f'MagDenseNetRegressor: {count_params(dn):,} params')

In [ ]:
# ── 6. Chargement des données ─────────────────────────────────
DATA_DIR = Path(DRIVE_DATA)
file_index = {p.name: p for p in DATA_DIR.rglob('*.npz')}
print(f'{len(file_index)} fichiers indexés')

df = pd.read_csv(CSV_PATH, sep=';')
df_pipe  = df[(df['label']==1) & df['width_m'].notna()].copy()
df_synth = df_pipe[~df_pipe['field_file'].str.startswith('real')].copy()
df_real  = df_pipe[df_pipe['field_file'].str.startswith('real')].copy()

def resolve(df_):
    paths, widths = [], []
    for _, row in df_.iterrows():
        if row['field_file'] in file_index:
            paths.append(file_index[row['field_file']])
            widths.append(row['width_m'])
    return paths, widths

synth_paths, synth_widths = resolve(df_synth)
real_paths,  real_widths  = resolve(df_real)

print(f'Synthétiques : {len(synth_paths)} | Réels : {len(real_paths)}')
print(f'width_m : min={min(synth_widths):.1f}m  max={max(synth_widths):.1f}m  mean={np.mean(synth_widths):.1f}m')

In [ ]:
# ── 7. Split train/val ────────────────────────────────────────
idx_tr, idx_val = train_test_split(range(len(synth_paths)), test_size=0.2, random_state=42)

train_ds = RegressionDataset([synth_paths[i] for i in idx_tr],  [synth_widths[i] for i in idx_tr],  augment=True)
val_ds   = RegressionDataset([synth_paths[i] for i in idx_val], [synth_widths[i] for i in idx_val], augment=False)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
# ── 8. Fonction d'entraînement ────────────────────────────────

def train_model(model_name='cnn', epochs=50, lr=1e-3, patience=10):

    model = (MagCNNRegressor() if model_name == 'cnn' else MagDenseNetRegressor()).to(DEVICE)
    print(f'\n{"="*55}')
    print(f'  Modèle : {model_name} | {count_params(model):,} params | device: {DEVICE}')
    print(f'{"="*55}')

    criterion = nn.HuberLoss(delta=1.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_mae      = float('inf')
    patience_ctr  = 0
    history       = []
    best_state    = None

    print(f'\n{"Epoch":>6} {"Train Loss":>12} {"Val Loss":>10} {"Val MAE":>10} {"LR":>10}')
    print('-' * 54)

    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        tr_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tr_loss += loss.item() * len(x)
        tr_loss /= len(train_ds)

        # Val
        model.eval()
        vl_loss = 0.0
        preds, tgts = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                p = model(x)
                vl_loss += criterion(p, y).item() * len(x)
                preds.extend(p.cpu().numpy())
                tgts.extend(y.cpu().numpy())
        vl_loss /= len(val_ds)
        mae_m = float(np.abs(np.expm1(np.array(preds)) - np.expm1(np.array(tgts))).mean())

        scheduler.step()
        lr_now = scheduler.get_last_lr()[0]
        history.append({'epoch': epoch, 'tr_loss': tr_loss, 'vl_loss': vl_loss, 'mae_m': mae_m})

        ok = '✓' if mae_m < 1.0 else '✗'
        print(f'{epoch:>6} {tr_loss:>12.4f} {vl_loss:>10.4f} {mae_m:>8.2f}m{ok}  {lr_now:>10.6f}')

        if mae_m < best_mae:
            best_mae     = mae_m
            patience_ctr = 0
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}
            print(f'         ↑ Meilleur modèle sauvegardé (MAE={mae_m:.2f}m)')
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f'\n⏹  Early stopping epoch {epoch}')
                break

    model.load_state_dict(best_state)
    ok = '✅' if best_mae < 1.0 else '❌'
    print(f'\n{ok} Best val MAE : {best_mae:.2f}m  (objectif: <1m)')
    return model, history, best_mae

In [ ]:
# ── 9a. Entraînement MagCNN ───────────────────────────────────
model_cnn, history_cnn, best_mae_cnn = train_model('cnn', epochs=50, lr=1e-3, patience=10)

In [ ]:
# ── 9b. Entraînement DenseNet (si CNN insuffisant) ────────────
# Décommenter si MAE CNN > 5m
# model_dn, history_dn, best_mae_dn = train_model('densenet', epochs=50, lr=5e-4, patience=10)

In [ ]:
# ── 10. Courbes d'apprentissage ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_cnn = [h['epoch']   for h in history_cnn]
loss_tr    = [h['tr_loss'] for h in history_cnn]
loss_vl    = [h['vl_loss'] for h in history_cnn]
mae_vals   = [h['mae_m']   for h in history_cnn]

axes[0].plot(epochs_cnn, loss_tr, label='Train Loss')
axes[0].plot(epochs_cnn, loss_vl, label='Val Loss')
axes[0].set_title('MagCNN — Huber Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_cnn, mae_vals, color='orange', label='Val MAE (m)')
axes[1].axhline(1.0, color='red', linestyle='--', label='Objectif 1m')
axes[1].set_title('MagCNN — MAE en mètres')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE (m)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/skipper_ndt/t2_learning_curves.png', dpi=150)
plt.show()
print('Figure sauvegardée dans ton Drive')

In [ ]:
# ── 11. Test sur les 51 données réelles ──────────────────────
if real_paths:
    real_ds     = RegressionDataset(real_paths, real_widths, augment=False)
    real_loader = DataLoader(real_ds, batch_size=32, shuffle=False, num_workers=2)

    model_cnn.eval()
    preds_r, tgts_r = [], []
    with torch.no_grad():
        for x, y in real_loader:
            preds_r.extend(model_cnn(x.to(DEVICE)).cpu().numpy())
            tgts_r.extend(y.numpy())

    preds_r_m = np.expm1(np.array(preds_r))
    tgts_r_m  = np.expm1(np.array(tgts_r))
    real_mae  = np.abs(preds_r_m - tgts_r_m).mean()

    ok = '✅' if real_mae < 1.0 else '❌'
    print(f'\n{ok} MAE sur réels : {real_mae:.2f}m  (objectif: <1m)')
    print(f'\n  {"Fichier":<40} {"Prédit":>8} {"Vrai":>8} {"Erreur":>8}')
    print('  ' + '-'*66)
    for fname, pred, true in zip([p.name for p in real_paths], preds_r_m, tgts_r_m):
        err = abs(pred - true)
        ok_i = '✓' if err < 1.0 else ''
        print(f'  {fname:<40} {pred:>7.1f}m {true:>7.1f}m {err:>7.1f}m {ok_i}')

In [ ]:
# ── 12. Sauvegarde du modèle dans Drive ───────────────────────
save_path = '/content/drive/MyDrive/skipper_ndt/task2_best_model.pt'
torch.save({
    'model_state': model_cnn.state_dict(),
    'model_name':  'cnn',
    'best_mae_m':  best_mae_cnn,
    'task':        't2_map_width',
    'log_space':   True,
}, save_path)
print(f'✅ Modèle sauvegardé : {save_path}')
print(f'   Télécharge-le et mets-le dans task2/checkpoints/best_model.pt')